# Processamento de dados

In [1]:
# Standard
import os

# Third-party
import pandas as pd
import numpy as np
import geopandas as gpd
import xgboost as xgb
import joblib

## Dataset 1: Clima Bahia

In [2]:
clima_bahia = pd.read_csv(
    "../resources/clima_bahia_hackathon(1).csv",
    sep=",",
    encoding="utf-8",
    on_bad_lines="skip",
    low_memory=False
)

clima_bahia.head(5)

,ESTACAO,DATA (YYYY-MM-DD),HORA (UTC),PRECIPITACAO TOTAL HORARIO (mm),"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSAO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSAO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (W/m2),"TEMPERATURA DO AR - BULBO SECO, HORARIA (C)",TEMPERATURA DO PONTO DE ORVALHO (C),TEMPERATURA MAXIMA NA HORA ANT. (AUT) (C),TEMPERATURA MINIMA NA HORA ANT. (AUT) (C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIRECAO HORARIA (gr)","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)"
0,A401,2021-01-01,0,0.0,1008.0,1008.2,1007.9,NaN,26.0,20.9,26.3,25.9,21.3,20.6,74.0,72.0,73.0,93.0,5.2,1.5
1,A401,2021-01-01,100,0.0,1008.0,1008.1,1008.0,NaN,25.9,20.9,26.2,25.8,21.0,20.8,74.0,73.0,74.0,68.0,5.2,1.1
2,A401,2021-01-01,200,0.0,1007.7,1008.1,1007.7,NaN,25.6,20.9,26.0,25.6,21.1,20.8,76.0,74.0,75.0,72.0,4.6,1.0
3,A401,2021-01-01,300,0.0,1007.9,1007.9,1007.7,NaN,25.6,21.1,25.7,25.5,21.3,20.9,77.0,75.0,76.0,57.0,4.3,0.9
4,A401,2021-01-01,400,0.0,1007.7,1007.9,1007.6,NaN,25.5,21.4,25.7,25.4,21.5,21.0,79.0,76.0,78.0,62.0,2.7,0.7


In [3]:
clima_bahia.shape

(5206752, 20)

## Pré-processamento

Remoção de linhas duplicadas e vazias, identificação e tratamento de valores faltando (NaN)

In [4]:
def basic_pre_processing(df: pd.DataFrame) -> pd.DataFrame:
    # Remover duplicatas
    df = df.drop_duplicates()

    # Remover linhas vazias
    df = df.dropna(how="all")

    return df

In [5]:
clima_bahia = basic_pre_processing(clima_bahia)
clima_bahia.head(5)

,ESTACAO,DATA (YYYY-MM-DD),HORA (UTC),PRECIPITACAO TOTAL HORARIO (mm),"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSAO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSAO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (W/m2),"TEMPERATURA DO AR - BULBO SECO, HORARIA (C)",TEMPERATURA DO PONTO DE ORVALHO (C),TEMPERATURA MAXIMA NA HORA ANT. (AUT) (C),TEMPERATURA MINIMA NA HORA ANT. (AUT) (C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIRECAO HORARIA (gr)","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)"
0,A401,2021-01-01,0,0.0,1008.0,1008.2,1007.9,NaN,26.0,20.9,26.3,25.9,21.3,20.6,74.0,72.0,73.0,93.0,5.2,1.5
1,A401,2021-01-01,100,0.0,1008.0,1008.1,1008.0,NaN,25.9,20.9,26.2,25.8,21.0,20.8,74.0,73.0,74.0,68.0,5.2,1.1
2,A401,2021-01-01,200,0.0,1007.7,1008.1,1007.7,NaN,25.6,20.9,26.0,25.6,21.1,20.8,76.0,74.0,75.0,72.0,4.6,1.0
3,A401,2021-01-01,300,0.0,1007.9,1007.9,1007.7,NaN,25.6,21.1,25.7,25.5,21.3,20.9,77.0,75.0,76.0,57.0,4.3,0.9
4,A401,2021-01-01,400,0.0,1007.7,1007.9,1007.6,NaN,25.5,21.4,25.7,25.4,21.5,21.0,79.0,76.0,78.0,62.0,2.7,0.7


In [6]:
clima_bahia.shape

(5206752, 20)

### Tratando colunas de hora e data

In [7]:
# Regularizando formato da hora
clima_bahia["HORA (UTC)"] = (clima_bahia["HORA (UTC)"] / 100).astype(int)

# Separando coluna de data em colunas de ano, mês e dia 
clima_bahia['datetime'] = pd.to_datetime(
    clima_bahia['DATA (YYYY-MM-DD)'] + ' ' + clima_bahia["HORA (UTC)"].astype(str) + ':00', 
    format='%Y-%m-%d %H:%M'
)

# Removendo colunas antigas
clima_bahia = clima_bahia.drop([
    "DATA (YYYY-MM-DD)",
    "HORA (UTC)"
], axis=1)

# Adicionando coluna separada para o ano para facilitar join com outros dataframes
clima_bahia["ano"] = clima_bahia["datetime"].dt.year
clima_bahia.head(5)

,ESTACAO,PRECIPITACAO TOTAL HORARIO (mm),"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSAO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSAO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (W/m2),"TEMPERATURA DO AR - BULBO SECO, HORARIA (C)",TEMPERATURA DO PONTO DE ORVALHO (C),TEMPERATURA MAXIMA NA HORA ANT. (AUT) (C),TEMPERATURA MINIMA NA HORA ANT. (AUT) (C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIRECAO HORARIA (gr)","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",datetime,ano
0,A401,0.0,1008.0,1008.2,1007.9,NaN,26.0,20.9,26.3,25.9,21.3,20.6,74.0,72.0,73.0,93.0,5.2,1.5,2021-01-01 00:00:00,2021
1,A401,0.0,1008.0,1008.1,1008.0,NaN,25.9,20.9,26.2,25.8,21.0,20.8,74.0,73.0,74.0,68.0,5.2,1.1,2021-01-01 01:00:00,2021
2,A401,0.0,1007.7,1008.1,1007.7,NaN,25.6,20.9,26.0,25.6,21.1,20.8,76.0,74.0,75.0,72.0,4.6,1.0,2021-01-01 02:00:00,2021
3,A401,0.0,1007.9,1007.9,1007.7,NaN,25.6,21.1,25.7,25.5,21.3,20.9,77.0,75.0,76.0,57.0,4.3,0.9,2021-01-01 03:00:00,2021
4,A401,0.0,1007.7,1007.9,1007.6,NaN,25.5,21.4,25.7,25.4,21.5,21.0,79.0,76.0,78.0,62.0,2.7,0.7,2021-01-01 04:00:00,2021


### Transformando coluna datetime em index temporariamente

In [8]:
clima_bahia = clima_bahia.set_index("datetime").sort_index()
clima_bahia.head(5)

,ESTACAO,PRECIPITACAO TOTAL HORARIO (mm),"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSAO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSAO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (W/m2),"TEMPERATURA DO AR - BULBO SECO, HORARIA (C)",TEMPERATURA DO PONTO DE ORVALHO (C),TEMPERATURA MAXIMA NA HORA ANT. (AUT) (C),TEMPERATURA MINIMA NA HORA ANT. (AUT) (C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIRECAO HORARIA (gr)","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",ano
datetime,,,,,,,,,,,,,,,,,,,
2000-05-13 00:00:00,A401,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,2000
2000-05-13 01:00:00,A401,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,2000
2000-05-13 02:00:00,A401,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,2000
2000-05-13 03:00:00,A401,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,2000
2000-05-13 04:00:00,A401,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,-9999.0,2000


### Filtrando por resultados de 2015 até 2021

In [9]:
clima_bahia = pd.DataFrame(clima_bahia.loc[clima_bahia.index.year >= 2015])
clima_bahia.head(5)

,ESTACAO,PRECIPITACAO TOTAL HORARIO (mm),"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSAO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSAO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (W/m2),"TEMPERATURA DO AR - BULBO SECO, HORARIA (C)",TEMPERATURA DO PONTO DE ORVALHO (C),TEMPERATURA MAXIMA NA HORA ANT. (AUT) (C),TEMPERATURA MINIMA NA HORA ANT. (AUT) (C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIRECAO HORARIA (gr)","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",ano
datetime,,,,,,,,,,,,,,,,,,,
2015-01-01,A438,0.0,1015.4,1015.4,1014.8,-9999.0,25.7,21.9,25.8,25.5,21.9,21.4,80.0,77.0,79.0,118.0,3.8,1.8,2015
2015-01-01,A437,0.0,1007.5,1007.5,1007.0,-9999.0,22.9,20.9,23.6,22.8,20.9,20.7,89.0,85.0,89.0,329.0,1.7,1.0,2015
2015-01-01,A422,0.0,1013.4,1013.4,1012.7,19.4,25.7,22.6,25.8,25.6,22.8,22.5,84.0,83.0,83.0,56.0,9.8,8.2,2015
2015-01-01,A436,0.0,978.4,978.5,977.7,-9999.0,26.0,17.7,27.3,25.4,17.7,16.5,62.0,52.0,60.0,127.0,7.2,3.6,2015
2015-01-01,A444,0.0,1004.8,1004.8,1004.2,-9999.0,21.8,20.5,22.5,21.8,20.9,20.5,93.0,90.0,92.0,253.0,1.1,0.3,2015


In [10]:
clima_bahia.shape

(2423016, 19)

### Determinando porcentagem de valores NaN por atributo

In [11]:
clima_bahia.isna().sum() / len(clima_bahia) * 100

ESTACAO                                                   0.000000
PRECIPITACAO TOTAL HORARIO (mm)                           7.019145
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     4.015079
PRESSAO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           4.024984
PRESSAO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          4.024984
RADIACAO GLOBAL (W/m2)                                   18.552787
TEMPERATURA DO AR - BULBO SECO, HORARIA (C)               3.954493
TEMPERATURA DO PONTO DE ORVALHO (C)                       5.286181
TEMPERATURA MAXIMA NA HORA ANT. (AUT) (C)                 3.964563
TEMPERATURA MINIMA NA HORA ANT. (AUT) (C)                 3.964522
TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (C)           5.311314
TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (C)           5.350150
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  5.283622
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                  5.327534
UMIDADE RELATIVA DO AR, HORARIA (%)                       5.27

### Tratando valores faltando

In [12]:
# Dropando linhas com NaN de todos os atributos menos Radiação Global
clima_bahia = clima_bahia.dropna(
    subset=[c for c in clima_bahia.columns if c != "RADIACAO GLOBAL (W/m2)"]
)
clima_bahia.shape

(2214513, 19)

In [13]:
rad = clima_bahia["RADIACAO GLOBAL (W/m2)"]
rad = rad.interpolate(method="time", limit=48)
rad = rad.fillna(
    clima_bahia
    .groupby([clima_bahia.index.day, clima_bahia.index.hour])["RADIACAO GLOBAL (W/m2)"]
    .transform("mean")
)
clima_bahia["RADIACAO_IMPUTED"] = rad
clima_bahia.head(5)

,ESTACAO,PRECIPITACAO TOTAL HORARIO (mm),"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSAO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSAO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (W/m2),"TEMPERATURA DO AR - BULBO SECO, HORARIA (C)",TEMPERATURA DO PONTO DE ORVALHO (C),TEMPERATURA MAXIMA NA HORA ANT. (AUT) (C),TEMPERATURA MINIMA NA HORA ANT. (AUT) (C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIRECAO HORARIA (gr)","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",ano,RADIACAO_IMPUTED
datetime,,,,,,,,,,,,,,,,,,,,
2015-01-01,A438,0.0,1015.4,1015.4,1014.8,-9999.0,25.7,21.9,25.8,25.5,21.9,21.4,80.0,77.0,79.0,118.0,3.8,1.8,2015,-9999.0
2015-01-01,A437,0.0,1007.5,1007.5,1007.0,-9999.0,22.9,20.9,23.6,22.8,20.9,20.7,89.0,85.0,89.0,329.0,1.7,1.0,2015,-9999.0
2015-01-01,A422,0.0,1013.4,1013.4,1012.7,19.4,25.7,22.6,25.8,25.6,22.8,22.5,84.0,83.0,83.0,56.0,9.8,8.2,2015,19.4
2015-01-01,A436,0.0,978.4,978.5,977.7,-9999.0,26.0,17.7,27.3,25.4,17.7,16.5,62.0,52.0,60.0,127.0,7.2,3.6,2015,-9999.0
2015-01-01,A444,0.0,1004.8,1004.8,1004.2,-9999.0,21.8,20.5,22.5,21.8,20.9,20.5,93.0,90.0,92.0,253.0,1.1,0.3,2015,-9999.0


### Resetar index

In [14]:
clima_bahia = clima_bahia.reset_index()
clima_bahia.head(5)

,datetime,ESTACAO,PRECIPITACAO TOTAL HORARIO (mm),"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSAO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSAO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (W/m2),"TEMPERATURA DO AR - BULBO SECO, HORARIA (C)",TEMPERATURA DO PONTO DE ORVALHO (C),TEMPERATURA MAXIMA NA HORA ANT. (AUT) (C),...,TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIRECAO HORARIA (gr)","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",ano,RADIACAO_IMPUTED
0,2015-01-01,A438,0.0,1015.4,1015.4,1014.8,-9999.0,25.7,21.9,25.8,...,21.9,21.4,80.0,77.0,79.0,118.0,3.8,1.8,2015,-9999.0
1,2015-01-01,A437,0.0,1007.5,1007.5,1007.0,-9999.0,22.9,20.9,23.6,...,20.9,20.7,89.0,85.0,89.0,329.0,1.7,1.0,2015,-9999.0
2,2015-01-01,A422,0.0,1013.4,1013.4,1012.7,19.4,25.7,22.6,25.8,...,22.8,22.5,84.0,83.0,83.0,56.0,9.8,8.2,2015,19.4
3,2015-01-01,A436,0.0,978.4,978.5,977.7,-9999.0,26.0,17.7,27.3,...,17.7,16.5,62.0,52.0,60.0,127.0,7.2,3.6,2015,-9999.0
4,2015-01-01,A444,0.0,1004.8,1004.8,1004.2,-9999.0,21.8,20.5,22.5,...,20.9,20.5,93.0,90.0,92.0,253.0,1.1,0.3,2015,-9999.0


## Dataset 2: Uso do solo em áreas urbanizadas

In [15]:
map_biomas = pd.read_excel(
    "../resources/MAPBIOMAS_BRAZIL-COVERAGE_STATISTICS-COL.10.1-MUNICIPALITIES_STATES_BIOMES.xlsx",
    sheet_name="COVERAGE_10.1"
)
map_biomas.head(5)

,country,biome,state,state_acronym,municipality,class_id,class_level_0,class_level_1,class_level_2,class_level_3,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Brasil,Amazônia,Acre,AC,Acrelândia,0,Undefined,6. Not Observed,6. Not Observed,6. Not Observed,...,14.178402,16.115742,16.379696,14.794822,14.970546,15.234995,14.002384,14.794761,14.178737,15.234836
1,Brasil,Amazônia,Acre,AC,Acrelândia,3,Natural,1. Forest,1.1. Forest Formation,1.1. Forest Formation,...,59622.392302,58090.821041,57786.236961,57181.401709,55739.605360,53993.083542,52182.020497,49368.132731,46946.282841,45632.716365
2,Brasil,Amazônia,Acre,AC,Acrelândia,6,Natural,1. Forest,1.4 Floodable Forest,1.4 Floodable Forest,...,1529.614323,1537.012126,1541.061285,1532.344358,1533.047789,1537.978636,1520.461675,1507.597594,1486.986055,1520.181600
3,Brasil,Amazônia,Acre,AC,Acrelândia,11,Natural,2. Non Forest Natural Formation,2.1. Wetland,2.1. Wetland,...,21.401012,24.130905,24.751571,28.273405,35.753658,46.056572,45.968447,24.835715,19.554040,21.051771
4,Brasil,Amazônia,Acre,AC,Acrelândia,12,Natural,2. Non Forest Natural Formation,2.2. Grassland,2.2. Grassland,...,54.703320,50.562474,52.763268,40.256034,65.980574,50.297927,86.857863,93.906317,78.226264,36.467106


In [16]:
map_biomas.shape

(78818, 51)

In [17]:
map_biomas = basic_pre_processing(map_biomas)
map_biomas.shape

(78818, 51)

### Filtrando df por estado e cidade

In [18]:
map_biomas = map_biomas.loc[
    (map_biomas["state"] == "Bahia") &
    (map_biomas["municipality"] == "Salvador")
]
map_biomas.head(5)

,country,biome,state,state_acronym,municipality,class_id,class_level_0,class_level_1,class_level_2,class_level_3,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
40700,Brasil,Mata Atlântica,Bahia,BA,Salvador,3,Natural,1. Forest,1.1. Forest Formation,1.1. Forest Formation,...,4783.132030,4807.550687,4723.293756,4678.810183,4647.595174,4695.741705,4629.801017,4553.129878,4510.136548,4520.689499
40701,Brasil,Mata Atlântica,Bahia,BA,Salvador,4,Natural,1. Forest,1.2. Savanna Formation,1.2. Savanna Formation,...,0.174364,0.174364,0.174364,0.174364,0.174364,0.174364,0.174364,0.174364,0.174364,0.174364
40702,Brasil,Mata Atlântica,Bahia,BA,Salvador,5,Natural,1. Forest,1.3. Mangrove,1.3. Mangrove,...,121.174955,127.543758,126.671353,125.188267,127.107581,127.805552,128.939674,119.953622,115.853331,114.108477
40703,Brasil,Mata Atlântica,Bahia,BA,Salvador,9,Antropic,3. Farming,3.3. Forest Plantation,3.3. Forest Plantation,...,3.663898,3.663898,3.663898,3.663898,3.663898,3.663898,3.663898,3.663898,3.663898,3.663898
40704,Brasil,Mata Atlântica,Bahia,BA,Salvador,11,Natural,2. Non Forest Natural Formation,2.1. Wetland,2.1. Wetland,...,27.298226,24.507082,24.332685,23.635223,24.158653,18.838283,16.222509,14.478070,15.001630,15.001633


### Pegando classes de interesse

In [19]:
target_classes = ["1.1. Forest Formation", "4.2. Urban Area"]
map_biomas = map_biomas[map_biomas["class_level_2"].isin(target_classes)]

### Removendo colunas pré-2015 e pós 2021 e transformando colunas de ano em uma só

In [20]:
map_biomas = map_biomas.melt(
    id_vars=["municipality", "class_level_2"],
    value_vars=[c for c in map_biomas.columns if type(c) == int and 2015 <= c <= 2021],
    var_name="ano",
    value_name="area"
)
map_biomas


,municipality,class_level_2,ano,area
0,Salvador,1.1. Forest Formation,2015,4783.132030
1,Salvador,4.2. Urban Area,2015,18634.252890
2,Salvador,1.1. Forest Formation,2016,4807.550687
3,Salvador,4.2. Urban Area,2016,18746.748639
4,Salvador,1.1. Forest Formation,2017,4723.293756
5,Salvador,4.2. Urban Area,2017,18838.750766
6,Salvador,1.1. Forest Formation,2018,4678.810183
7,Salvador,4.2. Urban Area,2018,18940.857876
8,Salvador,1.1. Forest Formation,2019,4647.595174
9,Salvador,4.2. Urban Area,2019,19043.925488


In [21]:
map_biomas = map_biomas.pivot(
    index="ano",
    columns="class_level_2",
    values="area"
).reset_index()
map_biomas = map_biomas.rename(columns={
    "1.1. Forest Formation": "Forest Formation",
    "4.2. Urban Area": "Urban Area"
})
map_biomas

class_level_2,ano,Forest Formation,Urban Area
0,2015,4783.132030,18634.252890
1,2016,4807.550687,18746.748639
2,2017,4723.293756,18838.750766
3,2018,4678.810183,18940.857876
4,2019,4647.595174,19043.925488
5,2020,4695.741705,19106.887038
6,2021,4629.801017,19220.856442


## Juntando dataframes 1 e 2

In [22]:
df_final = pd.merge(
    clima_bahia,
    map_biomas,
    how="inner",
    on="ano"
)
df_final.head(5)

,datetime,ESTACAO,PRECIPITACAO TOTAL HORARIO (mm),"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSAO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSAO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (W/m2),"TEMPERATURA DO AR - BULBO SECO, HORARIA (C)",TEMPERATURA DO PONTO DE ORVALHO (C),TEMPERATURA MAXIMA NA HORA ANT. (AUT) (C),...,UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIRECAO HORARIA (gr)","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",ano,RADIACAO_IMPUTED,Forest Formation,Urban Area
0,2015-01-01,A438,0.0,1015.4,1015.4,1014.8,-9999.0,25.7,21.9,25.8,...,80.0,77.0,79.0,118.0,3.8,1.8,2015,-9999.0,4783.13203,18634.25289
1,2015-01-01,A437,0.0,1007.5,1007.5,1007.0,-9999.0,22.9,20.9,23.6,...,89.0,85.0,89.0,329.0,1.7,1.0,2015,-9999.0,4783.13203,18634.25289
2,2015-01-01,A422,0.0,1013.4,1013.4,1012.7,19.4,25.7,22.6,25.8,...,84.0,83.0,83.0,56.0,9.8,8.2,2015,19.4,4783.13203,18634.25289
3,2015-01-01,A436,0.0,978.4,978.5,977.7,-9999.0,26.0,17.7,27.3,...,62.0,52.0,60.0,127.0,7.2,3.6,2015,-9999.0,4783.13203,18634.25289
4,2015-01-01,A444,0.0,1004.8,1004.8,1004.2,-9999.0,21.8,20.5,22.5,...,93.0,90.0,92.0,253.0,1.1,0.3,2015,-9999.0,4783.13203,18634.25289


## Treinamento do Modelo Preditivo (XGBoost)

In [23]:
# 1. Definir as variáveis preditoras (features) e a variável alvo (target)
features = [
    'RADIACAO_IMPUTED',
    'PRECIPITACAO TOTAL HORARIO (mm)',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'Forest Formation',
    'Urban Area'
]
target = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (C)'

# 2. Limpar os resíduos de -9999.0 e valores nulos que sobraram no df_final
df_model = df_final[df_final['RADIACAO_IMPUTED'] != -9999.0].copy()
df_model = df_model.dropna(subset=features + [target])

X = df_model[features]
y = df_model[target]

# 3. Treinar o modelo XGBoost Regressor
modelo_xgb = xgb.XGBRegressor(n_estimators=100, max_depth=5, random_state=42)
modelo_xgb.fit(X, y)

# 4. Exportar o modelo treinado para uso no Streamlit
joblib.dump(modelo_xgb, 'modelo_xgboost_calor.pkl')
print("Modelo treinado e salvo como 'modelo_xgboost_calor.pkl'")

ImportError: sklearn needs to be installed in order to use this module